In [1]:
from pathlib import Path
import re
import json
import datetime as dt

from itertools import chain

import geopandas as gpd
import pandas as pd

from glims_ingest_tools.writers import make_image_gmt, make_session_gmt

from utils import get_images_from_geodataframe

DATAPATH = Path.home() / "Data" / "GLIMS" / "submissions" / "0007_hartl_austria" / "GI5_v2"

SUBMISSON_PATH = DATAPATH / "submission"
FOR_INGEST_PATH = DATAPATH / "for_ingest"

In [2]:
OUTLINE_FILES = sorted([f for f in SUBMISSON_PATH.glob("*.geojson") if "vanishing" not in f.name])
OUTLINE_FILES

[PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Allgaeuer_Alpen_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Ankogel_Hochalmspitzgruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Dachstein_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Defreggergruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Glocknergruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Goldberggruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Granatspitzgruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Hochkoenig_GI5_proc2.geojson'),
 PosixPa

In [3]:
gdf = pd.concat([gpd.read_file(f) for f in OUTLINE_FILES], ignore_index=True)
gdf.head()

,id,year,name,data_type,src_date,source,image_ID1,flightstr,state,region,...,source_2,src_date_2,ROGI,van_fragm,nr_frag,median_elev,max_elev,min_elev,area,geometry
0,1800900.0,2023,Schwarzmilz Ferner,aerial image,20230905,\nPortal Tirol\nLBA\n,40641,0020,Tyrol,Allgaeuer_Alpen,...,,,0,None,1,2429.0,2454.0,2409.0,19941,"POLYGON ((170417.348 381998.047, 170417.041 38..."
1,4041.0,2023,Lassacher Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,KAGIS,A2023250_4522-5203_DOM,4522-5203,Carinthia,Ankogel_Hochalmspitzgruppe,...,,,0,None,3,2843.0,2930.0,2805.0,14582,"MULTIPOLYGON (((392996.945 349745.073, 392997...."
2,4042.0,2023,Plessnitz Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,KAGIS,"A2023250_4522-5203_DOM, A2023250_4522-5201_DOM","4522-5203, 4522-5201",Carinthia,Ankogel_Hochalmspitzgruppe,...,,,0,None,16,2890.0,3128.0,2692.0,63421,"MULTIPOLYGON (((393645.55 349594.969, 393645.6..."
3,4043.0,2023,Winkl Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,KAGIS,A2023250_4521-5101_DOM,4521-5101,Carinthia,Ankogel_Hochalmspitzgruppe,...,,,0,None,2,2894.0,3088.0,2552.0,338581,"MULTIPOLYGON (((398808.761 346194.866, 398805...."
4,4044.0,2023,Winkl Kees S,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,KAGIS,A2023250_4521-5101_DOM,4521-5101,Carinthia,Ankogel_Hochalmspitzgruppe,...,,,0,None,5,2616.0,2699.0,2533.0,3217,"MULTIPOLYGON (((397819.227 345323.725, 397819...."


## Homogenize Shapefile `source` names

Lea Hartl provided a text file containing key:value pairs for source name used on original shapefiles and full official names of these sources.  This was processed using `make_source_data_lookup_table.py` to create a JSON file containing a set of unique lookups.  This is used to homogenize source names.

In [5]:
with open('geojson_abbreviations_lookup.json', 'r') as f:
        source_lookup = json.load(f)
# source_lookup

Some edge cases remain.  These are fixed here.  Replace `source` column that contains list with LBA with just LBA

In [6]:
gdf["source"] = gdf["source"].where(gdf.source.str.contains("LBA") == False, "LBA")

Split `source` with two valid sources between `source` and `source_2`

In [7]:
tmp_index = gdf.source == 'Land Salzburg, Land Tirol'
gdf.loc[tmp_index, "source"] = "Land Tirol"
gdf.loc[tmp_index, "source_2"] = "Land Salzburg"

Homogenize source and source 2 using `source_lookup`

In [8]:
gdf["source"] = gdf.source.map(source_lookup)
gdf["source_2"] = gdf.source_2.map(source_lookup)

In [9]:
gdf

,id,year,name,data_type,src_date,source,image_ID1,flightstr,state,region,...,source_2,src_date_2,ROGI,van_fragm,nr_frag,median_elev,max_elev,min_elev,area,geometry
0,1800900.0,2023,Schwarzmilz Ferner,aerial image,20230905,Orthoimagery Province of Tyrol,40641,0020,Tyrol,Allgaeuer_Alpen,...,NaN,,0,None,1,2429.0,2454.0,2409.0,19941,"POLYGON ((170417.348 381998.047, 170417.041 38..."
1,4041.0,2023,Lassacher Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,DEM Hillshade Province of Carinthia,A2023250_4522-5203_DOM,4522-5203,Carinthia,Ankogel_Hochalmspitzgruppe,...,NaN,,0,None,3,2843.0,2930.0,2805.0,14582,"MULTIPOLYGON (((392996.945 349745.073, 392997...."
2,4042.0,2023,Plessnitz Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,DEM Hillshade Province of Carinthia,"A2023250_4522-5203_DOM, A2023250_4522-5201_DOM","4522-5203, 4522-5201",Carinthia,Ankogel_Hochalmspitzgruppe,...,NaN,,0,None,16,2890.0,3128.0,2692.0,63421,"MULTIPOLYGON (((393645.55 349594.969, 393645.6..."
3,4043.0,2023,Winkl Kees,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,DEM Hillshade Province of Carinthia,A2023250_4521-5101_DOM,4521-5101,Carinthia,Ankogel_Hochalmspitzgruppe,...,NaN,,0,None,2,2894.0,3088.0,2552.0,338581,"MULTIPOLYGON (((398808.761 346194.866, 398805...."
4,4044.0,2023,Winkl Kees S,"Lidar DEM/Hillshade 2023, DOD 2023-2010, Ortho...",202309XX,DEM Hillshade Province of Carinthia,A2023250_4521-5101_DOM,4521-5101,Carinthia,Ankogel_Hochalmspitzgruppe,...,NaN,,0,None,5,2616.0,2699.0,2533.0,3217,"MULTIPOLYGON (((397819.227 345323.725, 397819...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
794,9106.0,2022,Federbett Kees (>Tux),aerial image,20220912,Orthoimagery Province of Tyrol,5454,"0041, 0042",Tirol,Zillertaler_Alpen,...,NaN,,0,NaN,1,3090.0,3110.0,3068.0,9846,"POLYGON ((276418.602 355511.481, 276431.294 35..."
795,9107.0,2022,Schwarzbrunner Kees,aerial image,20220912,Orthoimagery Province of Tyrol,5500,"0041, 0042",Tirol,Zillertaler_Alpen,...,NaN,,0,NaN,1,2936.0,3036.0,2864.0,68639,"POLYGON ((275679.856 354543.3, 275684.221 3545..."
796,9108.0,2022,NN,aerial image,20220912,Orthoimagery Province of Tyrol,5498,0042,Tirol,Zillertaler_Alpen,...,NaN,,0,NaN,2,2905.0,2926.0,2863.0,1839,"MULTIPOLYGON (((275506.468 354252.76, 275489.8..."
797,9109.0,2022,Gefrorene Wand Kees E,aerial image,20220912,Orthoimagery Province of Tyrol,5496,"0042, 0043",Tirol,Zillertaler_Alpen,...,NaN,,0,NaN,2,2747.0,2902.0,2511.0,276743,"MULTIPOLYGON (((274711.283 353897.851, 274714...."


In [12]:
with open("unique_source_files.txt", "w") as fo:
    fo.write('\n'.join(gdf.source.unique()))

In [13]:
more unique_source_files.txt

Orthoimagery Province of Tyrol
DEM Hillshade Province of Carinthia
PlanetScope Orthoimagery
Orthoimagery Federal Office of Metrology and Surveying - data.bev.gv.at
Orthoimagery Province of Salzburg
Orthoimagery Province of Carinthia
UAV-borne photogrammetric survey by Geosphere Austria
Orthoimagery and DEM Hillshade Province of Salzburg
DEM Hillshade and Orthoimagery Province of Tyrol
Orthoimagery Province of Vorarlberg

In [23]:
gdf[gdf.source == "PlanetScope Orthoimagery"].image_ID1.loc[7]

'Multispectral_20230819_100506_49_2426_3B_AnalyticMS_8b_metadata_clip'

# Get Analysts

In [24]:
with open(FOR_INGEST_PATH / "subregions.json", "r") as f:
    subregions = json.load(f)

In [25]:
subregions 

[{'number': '1',
  'name': 'Allgaeuer Alpen',
  'contact': 'IGF-Glaziologie@oeaw.ac.at',
  'institute': 'Austrian Academy of Sciences, Institute for Interdisciplinary Mountain Research (OEAW-IGF)',
  'analysts': [{'analyst': {'first_name': 'Marie',
     'middle_name': 'Sophie',
     'surname': 'Kaucher'}}]},
 {'number': '2',
  'name': 'Ankogel Hochalmspitzgruppe',
  'contact': ' harald.zandler@uni-graz.at',
  'institute': 'Department of Geography and Regional Science, University of Graz',
  'analysts': [{'analyst': {'first_name': 'Andreas',
     'middle_name': '',
     'surname': 'Kellerer-Pirklbauer'}},
   {'analyst': {'first_name': 'Harald',
     'middle_name': '',
     'surname': 'Zandler'}}]},
 {'number': '3',
  'name': 'Dachstein',
  'contact': 'tobias.bolch@tugraz.at',
  'institute': 'Institute of Geodesy, Graz University of Technology',
  'analysts': [{'analyst': {'first_name': 'Martin',
     'middle_name': '',
     'surname': 'Verhounik'}},
   {'analyst': {'first_name': 'Tobias

In [44]:
for subregion in subregions:
    print(f'Region #: {subregion["number"]}  Region Name: {subregion["name"]}  Analysts: {parse_analysts(subregion["analysts"])}')

Region #: 1  Region Name: Allgaeuer Alpen  Analysts: Kaucher, Marie Sophie
Region #: 2  Region Name: Ankogel Hochalmspitzgruppe  Analysts: Kellerer-Pirklbauer, Andreas; Zandler, Harald
Region #: 3  Region Name: Dachstein  Analysts: Verhounik, Martin; Bolch, Tobias
Region #: 4  Region Name: Defreggergruppe  Analysts: Kaucher, Marie Sophie; Stocker-Waldhuber, Martin
Region #: 5  Region Name: Glocknergruppe  Analysts: Haut, Anna; Krippes, Julie; Otto, Jan-Cristoph
Region #: 6  Region Name: Goldberggruppe  Analysts: Hynek, Bernhard
Region #: 7  Region Name: Granatspitzgruppe  Analysts: Kogel, Ann Christin; Prinz, Rainer
Region #: 8  Region Name: Hochkoenig  Analysts: Haut, Anna; Krippes, Julie; Otto, Jan-Cristoph
Region #: 9  Region Name: Karnische Alpen  Analysts: Kellerer-Pirklbauer, Andreas; Zandler, Harald
Region #: 10  Region Name: Lechtaler Alpen  Analysts: Conzelmann, Svenja; Kaucher, Marie Sophie
Region #: 11  Region Name: Oetztaler Alpen  Analysts: Hartig, Anne; Kaucher, Marie; La

In [46]:
def analyst2str(analyst):
    s = f'{analyst["surname"]}, {analyst["first_name"]}'
    if analyst["middle_name"]:
        s += f' {analyst["middle_name"]}'
    return s

def parse_analysts(analysts):
    analyst_str = [analyst2str(a["analyst"]) for a in analysts]
    return '; '.join(analyst_str)

In [47]:
parse_analysts([{'analyst': {'first_name': 'Sina',
     'middle_name': '',
     'surname': 'Proelß'}},
   {'analyst': {'first_name': 'Marie',
     'middle_name': '',
     'surname': 'Kaucher'}}])

'Proelß, Sina; Kaucher, Marie'

In [77]:
analyst_list = []
for subregion in subregions:
    for a in subregion["analysts"]:
        analyst_str = analyst2str(a["analyst"])
        if not any([analyst_str in s for s in analyst_list]):
            analyst_list.append(analyst_str)
analyst_list
    

['Kaucher, Marie Sophie',
 'Kellerer-Pirklbauer, Andreas',
 'Zandler, Harald',
 'Verhounik, Martin',
 'Bolch, Tobias',
 'Stocker-Waldhuber, Martin',
 'Haut, Anna',
 'Krippes, Julie',
 'Otto, Jan-Cristoph',
 'Hynek, Bernhard',
 'Kogel, Ann Christin',
 'Prinz, Rainer',
 'Conzelmann, Svenja',
 'Hartig, Anne',
 'Lauria, Marcela Violeta',
 'Strudl, Markus',
 'Mayer, Christoph',
 'Helfricht, Kay',
 'Seiser, Bernd',
 'Akguen, Ayla Marie',
 'Schönleber, Lea',
 'Proelß, Sina']

In [78]:
for subregion in subregions:
    print(subregion["contact"])

IGF-Glaziologie@oeaw.ac.at
 harald.zandler@uni-graz.at
tobias.bolch@tugraz.at
IGF-Glaziologie@oeaw.ac.at
jan-christoph.otto@plus.ac.at
bernhard.hynek@geosphere.at
rainer.prinz@uibk.ac.at
jan-christoph.otto@plus.ac.at
 harald.zandler@uni-graz.at
IGF-Glaziologie@oeaw.ac.at
IGF-Glaziologie@oeaw.ac.at
IGF-Glaziologie@oeaw.ac.at
rainer.prinz@uibk.ac.at
IGF-Glaziologie@oeaw.ac.at
rainer.prinz@uibk.ac.at
IGF-Glaziologie@oeaw.ac.at
IGF-Glaziologie@oeaw.ac.at
jan-christoph.otto@plus.ac.at
IGF-Glaziologie@oeaw.ac.at
s.proelss@gmx.net
